In [1]:
import numpy as np
import xarray as xr
from cdo import *
from scipy.ndimage import distance_transform_edt
from osgeo import gdal, ogr, osr
from matplotlib import pyplot as plt
import pandas as pd
import pyproj as pp
from skimage.segmentation import flood_fill
from scipy.ndimage import label

In [2]:
# Define NEMO coastal mask of the area to parametrised
def get_coastal_msk(mask_in,lewp):
    nj_out=mask_in.shape[0]
    ni_out=mask_in.shape[1]

    if lewp:
        mask=np.zeros(shape=(nj_out,ni_out+2))
        mask[:,1:-1]=mask_in
        mask[:, 0]=mask[:,-2]
        mask[:,-1]=mask[:, 1]
        xslc=slice(0,ni_out)
    else:
        mask=mask_in
        xslc=slice(1,-1)
        
    mask_coast=np.zeros(shape=(nj_out,ni_out))
    mask_coast[1:-1,xslc]= ( mask[1:-1,1:-1] + 
                             mask[0:-2,1:-1] + mask[2::,1:-1] + mask[1:-1,0:-2] + mask[1:-1,2::] +
                             mask[0:-2,0:-2] + mask[2::,2::]  + mask[2::,0:-2]  + mask[0:-2,2::] ) * mask[1:-1,1:-1]
    mask_coast[(mask_coast > 1) & (mask_coast < 9)] = 10
    mask_coast[mask_coast!=10]=np.nan
    mask_coast=mask_coast.astype(np.float32)
    mask_coast[mask_coast==10]=1
    
    return mask_coast

In [3]:
def remove_small_ice_shelves(mask_isf, threshold):
    #1. Labeliser les régions connexes
    labeled_array, num_features = label(mask_isf)

    #2. Calculer la taille de chaque région
    sizes = np.bincount(labeled_array.ravel())
    sizes = sizes[1:]  # Ignorer le fond (label 0)

    #3. Trouver les labels des régions plus petites qu'un seuil
    small_regions = np.where(sizes < threshold)[0] + 1  # +1 car les labels commencent à 1

    #4. Créer un masque des ice shelves trop petits
    small_ice_shelves_mask = np.isin(labeled_array, small_regions)

    print('keep the {} largest ice shelves, remove the {} smallest ones'.format(num_features - len(small_regions), len(small_regions)))

    return mask_isf * (1 - small_ice_shelves_mask)  # Set small ice shelves to 0, keep the rest unchanged

In [4]:
# rasterise shapefile to raster
def rasterize_shapefile(geopackage_path, layer_name, attribute_field, x_min, y_max, width, height, pixel_size_x, pixel_size_y, no_data_value):
    # --- Step 1: Create an in-memory raster dataset ---
    driver = gdal.GetDriverByName("MEM")
    target_ds = driver.Create(
        "",  # Empty filename for in-memory dataset
        width,
        height,
        1,  # Number of bands
        gdal.GDT_Float64,  # Use Float64 to match your script
    )
    target_ds.SetGeoTransform((x_min, pixel_size_x, 0, y_max, 0, -pixel_size_y))

    # Set the CRS to EPSG:3031
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(3031)
    target_ds.SetProjection(srs.ExportToWkt())

    # Initialize the raster with no_data_value
    band = target_ds.GetRasterBand(1)
    band.SetNoDataValue(no_data_value)
    band.Fill(no_data_value)
    band.FlushCache()

    # --- Step 2: Rasterize the GeoPackage layer ---
    vector_ds = ogr.Open(geopackage_path)
    layer = vector_ds.GetLayerByName(layer_name)

    # Rasterize the layer, burning the attribute_field values
    gdal.RasterizeLayer(
        target_ds,
        [1],  # Band list (only band 1)
        layer,
        burn_values=[1],  # This is ignored if options=["ATTRIBUTE=..."] is used
        options=[f"ATTRIBUTE={attribute_field}"],
    )

    # Convert the GDAL dataset to an xarray DataArray ---
    return band.ReadAsArray(), target_ds.GetProjection()  # Flip vertically to match coordinate system


In [5]:
# Extrapolate basin IDs to coastal pixels using nearest neighbor interpolation
def extrapolate_data_nn(basin_mask, no_data_value):
    # Create a mask of valid (non-NaN) basin pixels
    valid_mask = (basin_mask != no_data_value)

    # Compute the distance from each pixel to the nearest valid basin pixel
    indices = distance_transform_edt(~valid_mask, return_distances=False, return_indices=True)

    # Get the indices of the nearest valid basin pixel for each pixel
    return basin_mask[tuple(indices)]

In [6]:
def compute_lat_lon_and_bounds(x_min, x_max, y_min, y_max, width, height, pixel_size): 
    # prepare variable needed for cdo 
    # Create coordinates (for dataarray)
    half_res = pixel_size / 2
    x_coords =   np.linspace(x_min + half_res, x_max - half_res, width)
    y_coords = - np.linspace(y_min + half_res, y_max - half_res, height)

    # Create 2D-coordinates
    meshx,meshy = np.meshgrid(x_coords, y_coords)

    # Create the lat and lon of the grid centers
    transformer_tolonlat = pp.Transformer.from_crs("EPSG:3031", "EPSG:4326", always_xy=True)

    # compute lat/lon
    print('    compute lat/lon')
    meshlon,meshlat = transformer_tolonlat.transform(meshx,meshy)

    # Original grid
    x_coords_bnds =   np.linspace(x_min, x_max, width  + 1)  # +1 for extended grid
    y_coords_bnds = - np.linspace(y_min, y_max, height + 1)  # +1 for extended grid

    # Create 2D-coordinates for extended grid
    meshx_bnds, meshy_bnds = np.meshgrid(x_coords_bnds, y_coords_bnds)

    # Create the lat and lon of the grid corners
    print('    compute lat/lon corners')
    meshlon_bnds,meshlat_bnds = transformer_tolonlat.transform(meshx_bnds, meshy_bnds)

    return x_coords, y_coords, meshlon, meshlat, meshlon_bnds, meshlat_bnds

In [7]:
# --- Parameters ---
pkgname="basins_for_param_v7"
geopackage_path = f"{pkgname}.gpkg"
layer_name = pkgname  # Name of the layer in the GeoPackage
attribute_field = "Id"               # Field to burn into the raster
no_data_value = -1.0

# Define the basin file on a 2 km BM grid
 - 1.: Rasterize the basin gpkg file on 2 km Bedmachine grid
 - 2.: Compute basin (and isf mask) array and extrapolate to be sure it overlap the NEMO coastal point once interpolated
 - 3.: Write the basin file with isf mask, basin and their extrapolation

In [8]:
# 1.: Rasterize shapefile to raster on 2km grid
   # 2km resolution gives 3335x3335 pixels, which is manageable in memory for interpolation and extrapolation
   # used only to find which basin is represneted in NEMO coastal pixels, not used for statistics
width, height = 3334, 3334          # Raster dimensions
x_min, y_min, x_max, y_max = -3334000.0, -3334000.0, 3334000, 3334000.0  # Adjust to your extent

pixel_size_x = (x_max - x_min) / width
pixel_size_y = (y_max - y_min) / height

data_basins, crs = rasterize_shapefile(geopackage_path, layer_name, attribute_field, x_min, y_max, width, height, pixel_size_x, pixel_size_y, no_data_value)

# 2.: Compute the basin ID array and extrapolate to overlap NEMO coastal pixels
    # build mask isf on the 2km grid
msk_isf_bm2km = np.where(data_basins < -1, 1, 0)               # This correspond to ice shelf
msk_isf_bm2km = np.where(data_basins > -1, 2, msk_isf_bm2km)   # This correspond to grounded ice basins
msk_isf_bm2km = np.where(data_basins == -1, 0, msk_isf_bm2km)  # This correspond to ocean

    # extrapolate ice shelf mask
msk_isf_bm2km_extrapolate = extrapolate_data_nn(msk_isf_bm2km, 0)
    # mask ish in basins
data_basins = np.where(data_basins <= -1, no_data_value, data_basins)

    # Extrapolate missing values using nearest neighbor ---
data_basins_extrapolate = extrapolate_data_nn(data_basins, no_data_value)

/Users/mathiotp/anaconda3/envs/ISFtoNEMO/lib/python3.11/site-packages/osgeo/gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [9]:
# 3.: Create basin, mask isf and extrapolation xarray dataset and save to netcdf
print('compute lat/lon and bounds for the 2km grid')
x_coords, y_coords, meshlon, meshlat, meshlon_bnds, meshlat_bnds = compute_lat_lon_and_bounds(x_min, x_max, y_min, y_max, width, height, pixel_size_x)

# Create the dataset with coordinates
ds_basins = xr.Dataset(
    coords={
        'y': y_coords,  # or use y_coords if you have them
        'x': x_coords,  # or use x_coords if you have them
        'lon':  xr.DataArray(data=meshlon, dims=['y', 'x'], name='lon',
                            attrs={
                                'standard_name': 'longitude',
                                'units': 'degrees_east',
                                'bounds': 'lon_bnds'
                            }
                ),
        'lat':  xr.DataArray(data=meshlat, dims=['y', 'x'], name='lat',
                            attrs={
                                'standard_name': 'latitude',
                                'units': 'degrees_north',
                                'bounds': 'lat_bnds'
                            }
                )
    },
    attrs={"crs": crs}
)

# Add lon_bnds and lat_bnds as variables (not coordinates)
ds_basins['lon_bnds'] = xr.DataArray(
                            data=np.array([meshlon_bnds[:-1,:-1], meshlon_bnds[1:,:-1], meshlon_bnds[1:,1:], meshlon_bnds[:-1,1:]]),
                            dims=['nvertex', 'y', 'x'],
                            attrs={'coordinates': 'lon lat'},
                            name='lon_bnds'
                        )
ds_basins['lat_bnds'] = xr.DataArray(
                            data=np.array([meshlat_bnds[:-1,:-1], meshlat_bnds[1:,:-1], meshlat_bnds[1:,1:], meshlat_bnds[:-1,1:]]),
                            dims=['nvertex', 'y', 'x'],
                            attrs={'coordinates': 'lon lat'},
                            name='lat_bnds'
                        )

# Add basins as a variable
ds_basins['basins']            = xr.DataArray(data_basins.astype(np.float64),             dims=['y', 'x'], attrs={'coordinates': 'lon lat'}, name='basins')
ds_basins['basins_extrapolate']= xr.DataArray(data_basins_extrapolate.astype(np.float64), dims=['y', 'x'], attrs={'coordinates': 'lon lat'}, name='basins_extrapolate')

# missing values should be -1
ds_basins['basins']             = ds_basins['basins'].where(            ds_basins['basins']             != no_data_value, other=no_data_value)
ds_basins['basins_extrapolate'] = ds_basins['basins_extrapolate'].where(ds_basins['basins_extrapolate'] != no_data_value, other=no_data_value)

ds_basins['mask']               = xr.DataArray(msk_isf_bm2km,             dims=['y', 'x'], attrs={'coordinates': 'lon lat'}, name='mask')
ds_basins['mask_extrapolate']   = xr.DataArray(msk_isf_bm2km_extrapolate, dims=['y', 'x'], attrs={'coordinates': 'lon lat'}, name='mask_extrapolate')

# save to netcdf
ds_basins = ds_basins.transpose('y','x','nvertex')
ds_basins.to_netcdf(f'{pkgname}_extrapolate.nc')

del meshlon, meshlat, meshlon_bnds, meshlat_bnds, x_coords, y_coords, data_basins, data_basins_extrapolate

compute lat/lon and bounds for the 2km grid
    compute lat/lon
    compute lat/lon corners


# Create Bedmachine grid at 500m resolution
Useful for CDO interpolation and BedMachine statistics
- 1.: Compute lat, lon and bounds
- 2.: Rasterize basins at 500m resolution (BedMachine grid)
- 3.: Save the 500m grid file
 

In [10]:
# 1.: Compute the lat/lon and bounds for the 500m grid
# original resolution to ice shelf statistics 
width_500m, height_500m = 13333, 13333

x_min_500m, y_min_500m, x_max_500m, y_max_500m = -3333250.0, -3333250.0, 3333250, 3333250.0

pixel_size_x_500m = (x_max_500m - x_min_500m) / width_500m
pixel_size_y_500m = (y_max_500m - y_min_500m) / height_500m

print('compute lat/lon and bounds for the 500m grid')
x_coords_500m, y_coords_500m, meshlon_500m, meshlat_500m, meshlon_bnds_500m, meshlat_bnds_500m = compute_lat_lon_and_bounds(x_min_500m, x_max_500m, y_min_500m, y_max_500m, width_500m, height_500m, pixel_size_x_500m)

compute lat/lon and bounds for the 500m grid
    compute lat/lon
    compute lat/lon corners


In [11]:
# 2.: Rasterise the original shapefile at 500m resolution to get the original basin mask at the same resolution as the bedmachine data set
# Rasterise the original shapefile at 500m resolution to get the original basin mask at the same resolution as the bedmachine data set
# computed here as NCO need a variable in the file to compute the interpolation.
data_basins_500m, crs = rasterize_shapefile(geopackage_path, layer_name, attribute_field, x_min_500m, y_max_500m, width_500m, height_500m, pixel_size_x_500m, pixel_size_y_500m, no_data_value)

# 3.: Create grid for cdo interpolation at 500m resolution and rasterise the original shapefile at 500m resolution to get the original basin mask at the same resolution as the bedmachine data set
# create grids for cdo interpolation at 500m resolution 
ds_grid_bm_500m = xr.Dataset(
    coords={
        'y': y_coords_500m,  # or use y_coords if you have them
        'x': x_coords_500m,  # or use x_coords if you have them
        'lon':  xr.DataArray(data=meshlon_500m, dims=['y', 'x'], name='lon',
                            attrs={
                                'standard_name': 'longitude',
                                'units': 'degrees_east',
                                'bounds': 'lon_bnds'
                            }
                ),
        'lat':  xr.DataArray(data=meshlat_500m, dims=['y', 'x'], name='lat',
                            attrs={
                                'standard_name': 'latitude',
                                'units': 'degrees_north',
                                'bounds': 'lat_bnds'
                            }
                ),
        # Add lon_bnds and lat_bnds as variables (not coordinates)
        'lon_bnds': xr.DataArray(
                            data=np.array([meshlon_bnds_500m[:-1,:-1], meshlon_bnds_500m[1:,:-1], meshlon_bnds_500m[1:,1:], meshlon_bnds_500m[:-1,1:]]),
                            dims=['nvertex', 'y', 'x'],
                            attrs={'coordinates': 'lon lat'},
                            name='lon_bnds'
                        ),
        'lat_bnds': xr.DataArray(
                            data=np.array([meshlat_bnds_500m[:-1,:-1], meshlat_bnds_500m[1:,:-1], meshlat_bnds_500m[1:,1:], meshlat_bnds_500m[:-1,1:]]),
                            dims=['nvertex', 'y', 'x'],
                            attrs={'coordinates': 'lon lat'},
                            name='lat_bnds'
                        )
    },
    attrs={"crs": crs}
)

ds_grid_bm_500m['basins'] = xr.DataArray(data_basins_500m, dims=['y', 'x'], attrs={'coordinates': 'lon lat'}, name='basins')

ds_grid_bm_500m.transpose('y','x','nvertex').to_netcdf(f'grid_bm_500m.nc')

del ds_grid_bm_500m, meshlon_bnds_500m, meshlat_bnds_500m, meshlon_500m, meshlat_500m, x_coords_500m, y_coords_500m

# NEMO coastal mask

In [12]:
# Read NEMO data ready for cdo processing
# Read domain_cfg.nc
#fNEMO_grid='domain_cfg_O2_nco_ncra_e3.nc'
#fNEMO_grid='domain_cfg_O1_nco_full_nco_simp.nc'
fNEMO_grid='nemo_domain_masked.nc'
ds_nemo_domain=xr.open_dataset(fNEMO_grid)

# get depth bins
NEMO_z_bins=[None]*len(ds_nemo_domain.e3t_1d.squeeze()+2)
NEMO_z_bins[0]=-np.inf
NEMO_z_bins[1:-1]=np.cumsum(ds_nemo_domain.e3t_1d.squeeze().values)[:]
NEMO_z_bins[-1]=np.inf

# get NEMO 
## ocean mask
NEMO_mask=ds_nemo_domain.bottom_level.squeeze()
NEMO_mask=np.where(NEMO_mask>0,1,0)

## NEMO Antarctic mask
NEMO_surface_mask = np.where( ds_nemo_domain.top_level > 1, 0, NEMO_mask )
NEMO_ANT_mask = flood_fill(NEMO_surface_mask, (0, 0), -1)
NEMO_mask = np.where( NEMO_ANT_mask >= 0, 1, NEMO_mask)

ds_nemo_domain['isf_mask']=xr.where((ds_nemo_domain.bottom_level > 0) & (ds_nemo_domain.top_level > 1), 1, 0)

# get coastal mask of NEMO grid
NEMO_mask_param_basins=get_coastal_msk(NEMO_mask,True)
ds_nemo_domain['param_mask']=xr.DataArray(NEMO_mask_param_basins, dims=['y', 'x'], attrs={'coordinates': 'lon lat'}, name='param_mask')

ds_nemo_domain.to_netcdf('nemo_domain.nc')

In [13]:
# Compute cdo interpolation BM to NEMO grid (largest basin fraction (remaplaf) => works for ORCA2 but not for eORCA1 !!!!!)
cdo=Cdo()
NEMO_basins=cdo.remapnn(f'{fNEMO_grid}',input=ds_basins, returnArray='basins_extrapolate').squeeze()

In [14]:
# get map of basin number for each coastal cell
NEMO_mask_param_basins=np.where((NEMO_mask_param_basins >= 0), NEMO_basins, -1)

# Define the Bedmachine mask and draft
- 1.: Read BedMachine data and draft and mask isf
- 2.: Define list of NEMO coastal cells where we will apply the parametrisation

In [15]:
# Read BedMachine data and draft and mask isf
    # Get BM data (mask, draft, ...)
cfile_bm="NSIDC-0756_BedMachineAntarctica_19700101-20191001_V04.1.nc"
ds = xr.open_dataset(cfile_bm)

    # Calculer le draft : - (surface - thickness)
draft = - (ds.surface - ds.thickness)
    # Masquer les valeurs invalides (où surface ou thickness = -9999)
draft_isf = draft.where((ds.surface != -9999) & (ds.thickness != -9999) & (ds.mask == 3)).values

# mask = ds.mask.where(ds.mask == 3).values  # Mask pour les bassins valides
    # Compute ice shelf mask and its extrapolation. 
    # => should we remove the tiny tiny ice shelf in BM (the one of few grid cell without name all along the coast)?
mask_isf = np.where(ds.mask==3,1,0)
mask_isf = remove_small_ice_shelves(mask_isf, threshold=49)  # Remove ice shelves smaller than 49 pixels (12.25 km2 at 500m resolution) (about the 300 largest ice shelves in BM)

    # extrapolate mask isf / grd => used to define which NEMO coastal cell are ice shelf cells
mask = xr.where( ( (ds.mask == 0) | (ds.mask == 4) ), 0, ds.mask)  # set ocean and Vostock to 0
mask = np.where( ( (ds.mask == 2) | (ds.mask == 3) ), 2, mask)     # set ice free and ice shelf to grounded (2)
mask = np.where(mask_isf == 1, 3, mask)                            # set the biggest ice shelf to floating (3)

# extrapolate data
mask_extrapolate = extrapolate_data_nn(mask, 0)
mask_isf_extrapolate = np.where(mask_extrapolate==3,1,0)

keep the 275 largest ice shelves, remove the 4502 smallest ones


# Define all NEMO coastal cell that contains ice shelves
 - 1.: Set a unique id in every NEMO coastal cell
 - 2.: Interpolate to BM 500m
 - 3.: Mask by the extrapolated ice shelf mask
 - 4.: retreive the remaining list of cells amd mask the NEMO array

In [16]:
# 3.: Define list of NEMO coastal cells where we will apply the parametrisation
    # For each NEMO coastal cell we assign a cell number (-1 elsewhere)
lenarray = NEMO_mask_param_basins.shape[0]*NEMO_mask_param_basins.shape[1]
arr = np.arange(lenarray)
NEMO_icell=arr.reshape(NEMO_mask_param_basins.shape)
NEMO_icell=np.where(NEMO_mask_param_basins>=0,NEMO_icell,-1)
ds_nemo_domain['param_icell']=xr.DataArray(NEMO_icell, dims=['y', 'x'], attrs={'coordinates': 'lon lat'}, name='param_icell')


In [17]:
# 4.: Define list of NEMO coastal cells that correspond to an ice shelf in BM
    ## Interpolate NEMO coastal cell number on BM grid
BM_NEMO_icell=cdo.remapnn('grid_bm_500m.nc',input=ds_nemo_domain.param_icell, returnArray='param_icell', options='-f nc4 -v')

    ## mask with BM isf mask
BM_NEMO_icell = BM_NEMO_icell * mask_isf_extrapolate

    ## retrieve list of icell
lst_icell = list(set(BM_NEMO_icell[BM_NEMO_icell > 0].astype(int)))

    ## mask all cell in NEMO that are not in this list (ie coastal cell is adjacent to grounded area or ice free area)
NEMO_mask_param_basins_isf = np.where( np.isin(NEMO_icell,np.array(list(lst_icell))), NEMO_mask_param_basins, -1)

    ## Check and correct the NEMO mask of isf basin id
        ### Some basin may have desapeared in the process. If this is the case, we restore the original basin Id coast cells for the missing basin Id
        ### Original basins list in the NEMO mask of coastal cells
NEMO_basins_list = np.unique(NEMO_mask_param_basins[NEMO_mask_param_basins >= 0]).astype(int)

        ### List after masking the non isf cells
NEMO_isf_param_list = np.unique(NEMO_mask_param_basins_isf[NEMO_mask_param_basins_isf >= 0]).astype(int)

        ### sanity check and correction if needed
for id in NEMO_basins_list:
    if id not in NEMO_isf_param_list:
        print(f'{id} missing after mask by ice shelf mask extrapolation')
        NEMO_mask_param_basins_isf = np.where(NEMO_mask_param_basins == id, NEMO_mask_param_basins, NEMO_mask_param_basins_isf)
print()
NEMO_isf_param_list = np.unique(NEMO_mask_param_basins_isf[NEMO_mask_param_basins_isf >= 0]).astype(int)
for id in NEMO_basins_list:
    if id not in NEMO_isf_param_list:
        print(f'{id} still missing after mask by ice shelf mask extrapolation')

    ## NEMO_mask_param_basins ready for parametrisation: it contains the basin ID for each coastal cell that is adjacent to an ice shelf in BM, and -1 elsewhere
NEMO_mask_param_basins = NEMO_mask_param_basins_isf

# Correct the basin definition on BedMachine 500m grid
Some basins in the original shapefile may not be present anymore in the NEMO mask of the basin to parametrise. We need to:
 - 1.: remove them from the original basin Id array
 - 2.: extrapolate the BedMachine basin Id array

In [18]:
# Extrapolate missing values using nearest neighbor --- => is this one really useful as we do it anyway in the next cell. 
# We can directly use the original rasterised basin mask at 500m resolution as we will mask it in the next step based on the list of basins available in NEMO
ANT_basins_from_NEMO = extrapolate_data_nn(data_basins_500m, no_data_value)

# 1.: remove them from the original basin Id array
    # Define the new basin mask based on the list of available basins in NEMO
ANT_basins_list = np.unique(data_basins_500m).astype(int)
for basin in set(ANT_basins_list) - set(NEMO_basins_list):
    ANT_basins_from_NEMO[ANT_basins_from_NEMO == basin] = -1

# 2.: extrapolate the basin data on the bm grid to fill the masked values (at the end only basin present in NEMO will be kept)
ANT_basins_from_NEMO_extrapolate = extrapolate_data_nn(ANT_basins_from_NEMO, no_data_value)

# Define all NEMO coastal cell that contains ice shelves
 - 1.: Set a unique id in every NEMO coastal cell
 - 2.: Interpolate to BM 500m
 - 3.: Mask by the extrapolated ice shelf mask
 - 4.: retreive the remaining list of cells amd mask the NEMO array

In [19]:
# Check basin Id list presence in the extrapolated data
basin_mask = np.where(mask_isf == 1, ANT_basins_from_NEMO_extrapolate, np.nan)

# list of unique basin IDs in the gpkg rasterized data (extrapolated)
basin_ids = np.unique(ANT_basins_from_NEMO_extrapolate)

# check if all basin IDs in the mask are present in the extrapolated data
basin_ids_mask = np.unique(basin_mask[~np.isnan(basin_mask)])

# check if all basin IDs in the mask are present in the extrapolated data
all_present = np.all(basin_ids_mask == basin_ids)
print(f"All basin IDs in the mask are present in the extrapolated data: {all_present}")

All basin IDs in the mask are present in the extrapolated data: True


# Compute statistics of the draft distribution

In [20]:
# 1.: Interpolation isf mask from NEMO to BedMachine grid 
    # NEMO isf fraction on bm grid
    # Used to weight the area in the ice shelf draft distribution
    # conservative interpolation is needed but it does not work with eORCA1
BM_NEMO_isf_fraction=cdo.remapnn('grid_bm_500m.nc',input=ds_nemo_domain.isf_mask, returnArray='isf_mask', options='-f nc4 -v')

In [21]:
# Create a DataFrame for grouping and compute percentiles
# define area array (500m x 500m) for each pixel, same shape as draft_isf
area = np.ones_like(draft_isf)*500*500
area = np.where(mask_isf == 1, area, np.nan)

# weight area by the NEMO_isf_fraction on the BedMachine Grid
area = area * (1 - BM_NEMO_isf_fraction)

# Create a DataFrame for grouping and compute percentiles
df = pd.DataFrame({
    'basin_id': basin_mask.flatten(),
    'draft': draft_isf.flatten(),
    'area': area.flatten()
}).dropna()

# Define depth bins (no labels)
depth_bins = NEMO_z_bins
# Assign depth to bins (returns bin indices)
df['draft_bin'] = pd.cut(df['draft'], bins=depth_bins, labels=False)

In [22]:
# Compute area distribution (counts)
area_distribution = df.groupby(['basin_id', 'draft_bin'])['area'].sum().unstack(fill_value=0)

# Group by basin_id and compute percentiles
result = df.groupby('basin_id')['draft'].agg(['min', 'max', lambda x: np.nanpercentile(x, 25), lambda x: np.nanpercentile(x, 99)])
result.columns = ['min', 'max', 'zmin', 'zmax']

In [23]:
# retrieve the percentiles into separate lists
zmin_list = result['zmin'].tolist()  # 25th percentile
zmax_list = result['zmax'].tolist()  # 99th percentile
area_dist = np.array(area_distribution).T # Transpose to have draft bins as rows and basin_ids as columns

In [24]:
# sanity check : total distribution area should mach bedmachine area
total_area_distribution = area_dist.sum()
total_area_bedmachine = np.nansum(area)
print(f"Total area from distribution: {total_area_distribution}, Total area from bedmachine: {total_area_bedmachine}")

Total area from distribution: 677047250000.0, Total area from bedmachine: 677047250000.0


In [25]:
ds_coords = xr.open_dataset('coordinates_cdo.nc')
NEMO_area=ds_coords.e1t*ds_coords.e2t*ds_nemo_domain['isf_mask']

area_isf = np.ones_like(draft_isf)*500*500
area_isf = np.where(mask_isf == 1, area_isf, np.nan)

print(f"Total area from distribution + NEMO: {np.nansum(area_isf)/1e6}, Total area from bedmachine + NEMO: {(total_area_bedmachine+NEMO_area.sum().values)/1e6}")

Total area from distribution + NEMO: 1513208.125, Total area from bedmachine + NEMO: 1543094.8572861832


# Save output

In [26]:
# write netcdf with zmin(Id), zmax(Id), area distribution (z, Id), z_bounds(z,2), basin_Id(Id), NEMO_mask_param_basins(x,y), lon, lat, lon_bnds, lat_bnds
# Create the dataset with coordinates
ds_out = xr.Dataset(
    coords={
        'id': basin_ids,  # basin IDs   
        'lon': ds_nemo_domain.lon,  # longitude of grid centers from NEMO domain
        'lat': ds_nemo_domain.lat   # latitude of grid centers from NEMO domain
    },
)

# Add lon_bnds and lat_bnds as variables (not coordinates)
ds_out['lon_bnds'] = ds_nemo_domain.lon_bnds
ds_out['lat_bnds'] = ds_nemo_domain.lat_bnds
ds_out['z_bounds'] = xr.DataArray(data=np.array([depth_bins[:-1], depth_bins[1:]]).T, dims=['z', 'bound'], name='z_bounds')

# Add basins as a variable
ds_out['zmin'] = xr.DataArray(zmin_list, dims=['id'], name='zmin')
ds_out['zmax'] = xr.DataArray(zmax_list, dims=['id'], name='zmax')
ds_out['area_distribution'] = xr.DataArray(area_dist, dims=['z_bin', 'id'], name='area_distribution')
ds_out['basins']            = xr.DataArray(NEMO_mask_param_basins, dims=['y', 'x'], attrs={'coordinates': 'lon lat'}, name='basins')

# attrs
ds_out['lon'].attrs={
    'standard_name': 'longitude',
    'units': 'degrees_east',
    'bounds': 'lon_bnds'
}

ds_out['lat'].attrs={
    'standard_name': 'latitude',
    'units': 'degrees_north',
    'bounds': 'lat_bnds'
}

ds_out['zmin'].attrs={
    'long_name': '25th percentile of draft',
    'units': 'm'
}

ds_out['zmax'].attrs={
    'long_name': '99th percentile of draft',
    'units': 'm'
}

ds_out['area_distribution'].attrs={
    'long_name': 'Area distribution of draft bins',
    'units': 'm^2',
    'description': 'Area in m^2 for each draft bin and basin ID'
}

ds_out['z_bounds'].attrs={
    'long_name': 'Depth bin edges',
    'units': 'm',
    'description': 'Edges of depth bins used for area distribution'
}

ds_out['basins'].attrs={
    'long_name': 'Basin ID',
    'description': 'Basin ID for each grid cell'
}

ds_out.attrs={
    'title': 'Basin statistics for Antarctic ice shelves',
    'source': f'Derived from {cfile_bm} and custom basin delineation ({geopackage_path} on {fNEMO_grid} grid)',
    'description': 'Contains percentiles of draft and area distribution for each basin ID, along with the basin mask on the NEMO grid.'}

# save to netcdf
ds_out.to_netcdf('out_O1.nc')
